# Agente FAPESP — panorama unificado de notícias e instituições

Este Colab reúne **os coletores originais da pasta compartilhada**, os CSVs produzidos na sessão e o **histórico semanal acumulado no GitHub**. A cada execução ele lê as configurações atuais dos notebooks originais, coleta as fontes acessíveis, elimina URLs repetidas e monta um painel com filtros, tabelas e gráficos. Os notebooks originais continuam no lugar.

**Como usar:** execute todas as células. Autorize o acesso ao Google Drive quando o Colab solicitar; essa autorização permite ler os notebooks da pasta e gravar os resultados nela. A coleta completa de dezenas de portais pode levar vários minutos. Para um primeiro teste, altere `MODO = "rapido"` na célula de configuração.

**Leitura responsável:** a data de publicação nunca é substituída pela data da coleta. Eventos da Biblioteca, páginas de referência e registros sem data são identificados à parte. Um portal bloqueado aparece no diagnóstico e não é contado como ausência de notícias. Os temas são pistas encontradas **nos títulos**, não conclusões sobre os textos completos.

[Notebooks originais no Drive](https://drive.google.com/drive/folders/1SpKKCSz99_V6OsMLzN7XNYFHXXullzvu) · [Abrir esta versão do GitHub no Colab](https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Analise/agente_fapesp_unificado.ipynb) · [Repositório e metodologia](https://github.com/anamacao/FAPESP-PIBIC-scrapping)

## 1. Preparar o ambiente

O código de análise fica versionado no mesmo repositório dos notebooks. O Colab busca a versão atual a cada nova sessão; não precisa copiar funções entre 26 arquivos.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests>=2.32,<3", "beautifulsoup4>=4.12,<5", "pandas>=2.2,<3",
                "plotly>=5.24,<7", "ipywidgets>=8,<9", "google-api-python-client>=2,<3"], check=True)
print("Dependências prontas.")

## 2. Configurar a rodada

`completo` usa o número de páginas definido em cada notebook original; `rapido` consulta só a primeira página de cada fonte. Os resultados são salvos em uma subpasta **Resultados - agente FAPESP** dentro da pasta compartilhada. A execução semanal do GitHub alimenta quatro fontes continuamente; os demais portais são atualizados quando este Colab é executado.

In [ ]:
MODO = "completo"                 # "completo" ou "rapido"
COLETAR_NOTEBOOKS_DRIVE = True    # usa os coletores originais da pasta
INCLUIR_CSVS_DA_SESSAO = True    # aproveita saídas de outros Colabs nesta sessão
SALVAR_RESULTADOS_NO_DRIVE = True
TRABALHADORES = 3                # até 4; respeite limites dos portais

from pathlib import Path
import json, sys, subprocess
from IPython.display import display, HTML, clear_output

PASTA_DRIVE = "1SpKKCSz99_V6OsMLzN7XNYFHXXullzvu"
SAIDA = Path("/content/fapesp-data/agente_unificado") if Path("/content").exists() else Path.cwd() / "fapesp-data/agente_unificado"
REPO = Path("/content/fapesp-agent-repo") if Path("/content").exists() else Path.cwd() / "fapesp-agent-repo"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/anamacao/FAPESP-PIBIC-scrapping.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
import importlib.util
especificacao = importlib.util.spec_from_file_location("fapesp_agent", REPO / "scripts/fapesp_agent.py")
agent = importlib.util.module_from_spec(especificacao)
sys.modules[especificacao.name] = agent
especificacao.loader.exec_module(agent)
print("Código atualizado. Saída:", SAIDA)

## 3. Conectar à pasta e conferir as fontes

O agente lê a versão atual de cada notebook. Quando dois arquivos apontam para a mesma listagem, coleta uma vez e registra a duplicidade no diagnóstico. Os dois notebooks antigos de gráficos são substituídos pelo painel abaixo.

In [ ]:
servico_drive = None
erros_acesso = []
if COLETAR_NOTEBOOKS_DRIVE or SALVAR_RESULTADOS_NO_DRIVE:
    try:
        from google.colab import auth
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build
        credenciais, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
        servico_drive = build("drive", "v3", credentials=credenciais, cache_discovery=False)
        print("Drive conectado.")
    except Exception as exc:
        erros_acesso.append({"notebook": "pasta do Drive", "status": "access_failed", "error": str(exc)[:350]})
        print("Acesso ao Drive indisponível nesta sessão:", exc)

notebooks = []
if COLETAR_NOTEBOOKS_DRIVE and servico_drive:
    try:
        notebooks, falhas_download = agent.list_drive_notebooks(servico_drive, PASTA_DRIVE)
        erros_acesso.extend(falhas_download)
        print(f"{len(notebooks)} notebooks coletores encontrados na pasta.")
    except Exception as exc:
        erros_acesso.append({"notebook": "pasta do Drive", "status": "list_failed", "error": str(exc)[:350]})
        print("Falha ao listar os notebooks:", exc)
elif COLETAR_NOTEBOOKS_DRIVE:
    print("A análise seguirá com o histórico semanal e os CSVs da sessão; a cobertura do Drive ficará incompleta.")

## 4. Coletar e consolidar

Cada fonte retorna registros e um diagnóstico separado. Falhas não apagam registros históricos do CSV semanal. O notebook do Parlamento distingue notícias oficiais de eventos da Biblioteca; links de catálogos institucionais não são transformados em notícias.

In [ ]:
import requests
registros = []
diagnosticos = list(erros_acesso)
if notebooks:
    novos, saude = agent.collect_drive(notebooks,
        max_pages=1 if MODO == "rapido" else None, workers=TRABALHADORES)
    registros.extend(novos)
    diagnosticos.extend(saude)

try:
    resposta = requests.get(agent.WEEKLY_CSV, timeout=40)
    resposta.raise_for_status()
    historico = agent.weekly_rows(resposta.content)
    registros.extend(historico)
    for fonte in sorted({r["source_id"] for r in historico}):
        total = sum(r["source_id"] == fonte for r in historico)
        diagnosticos.append({"notebook": f"workflow semanal: {fonte}", "source_id": fonte,
                              "status": "historical", "records": total, "pages_ok": 0,
                              "pages_requested": 0, "errors": []})
    print(f"Histórico semanal: {len(historico)} registros acumulados.")
except Exception as exc:
    diagnosticos.append({"notebook": "workflow semanal no GitHub", "status": "failed",
                          "records": 0, "error": f"{type(exc).__name__}: {exc}"[:350]})
    print("Histórico semanal indisponível:", exc)

if INCLUIR_CSVS_DA_SESSAO:
    arquivos_sessao = agent.session_csv_rows(Path("/content/fapesp-data") if Path("/content").exists() else Path.cwd() / "fapesp-data")
    registros.extend(arquivos_sessao)
    print(f"CSVs de outros Colabs nesta sessão: {len(arquivos_sessao)} linhas.")

dados, qualidade = agent.normalize_records(registros)
if dados.empty:
    raise RuntimeError("Nenhum registro válido disponível. Confira a autenticação do Drive e os diagnósticos acima.")
print(f"Consolidação: {qualidade['valid']} registros válidos; {qualidade['unique']} URLs únicas; {qualidade['duplicates']} duplicações removidas.")
display(dados["record_type"].value_counts().rename_axis("tipo").reset_index(name="registros"))
display(agent.source_health(dados, diagnosticos))

## 5. Qualidade e cobertura da amostra

O total mostra URLs únicas, após retirar repetições entre notebooks, arquivos da sessão e histórico do GitHub. A proporção de datas reconhecidas e a tabela de falhas ajudam a decidir quais comparações podem ser interpretadas.

In [ ]:
print("URLs únicas:", len(dados), "| Datas reconhecidas:", int(dados["published_date"].notna().sum()))
display(dados.groupby(["region", "country", "record_type"]).size().reset_index(name="registros")
        .sort_values("registros", ascending=False))
display(agent.source_health(dados, diagnosticos))

## 6. Evolução, fontes e temas

O volume mensal indica o que foi **coletado**, não toda a produção institucional. As comparações de temas usam percentuais de títulos por região, com o denominador e um aviso para amostras pequenas.

In [ ]:
visao_inicial = agent.build_dashboard(dados, diagnosticos, days=365)
display(visao_inicial["regional"])
display(visao_inicial["topics"])
for figura in visao_inicial["figures"]:
    titulo = figura.layout.title.text or ""
    if titulo.startswith(("Registros por fonte", "Publicações coletadas", "Temas mencionados")):
        figura.show()

## 7. Palavras-chave e evidências

Cada termo é contado uma vez por título. O mapa de calor mostra a porcentagem de títulos com o termo em cada país da fonte; ele permite comparar amostras de tamanhos diferentes. A tabela com URLs permite conferir os textos originais.

In [ ]:
display(visao_inicial["keywords"])
for figura in visao_inicial["figures"]:
    titulo = figura.layout.title.text or ""
    if titulo.startswith(("Termos mais", "Palavras-chave por país", "Prevalência temática", "Títulos que mencionam")):
        figura.show()
display(visao_inicial["evidence"].head(30))

## 8. Painel interativo

Filtre por período, país da fonte, instituição, tipo de registro e palavra no título. Os gráficos mostram volumes mensais, proporções temáticas, termos frequentes e coocorrências. A tabela de evidências mantém **título e URL** para você voltar ao documento original.

In [ ]:
import ipywidgets as widgets
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

periodo = widgets.Dropdown(options=[("Últimos 7 dias", 7), ("Últimos 30 dias", 30),
                                   ("Últimos 90 dias", 90), ("Último ano", 365),
                                   ("Todo o histórico", None)], value=365, description="Período")
paises = widgets.SelectMultiple(options=sorted(dados["country"].unique()),
                                description="País da fonte", rows=6, layout=widgets.Layout(width="360px"))
fontes = widgets.SelectMultiple(options=[(f"{sid} · {len(g)}", sid) for sid, g in dados.groupby("source_id")],
                                description="Fonte", rows=8, layout=widgets.Layout(width="400px"))
outros_tipos = widgets.Checkbox(value=False, description="Incluir eventos, referências e registros sem data")
busca = widgets.Text(description="No título", placeholder="Ex.: proteção de dados", layout=widgets.Layout(width="440px"))
botao = widgets.Button(description="Atualizar gráficos", button_style="primary", icon="bar-chart")
botao_exportar = widgets.Button(description="Exportar seleção", icon="download")
painel = widgets.Output()
estado = {}

def mostrar(_=None):
    resultado = agent.build_dashboard(dados, diagnosticos, days=periodo.value,
        countries=list(paises.value), source_ids=list(fontes.value),
        include_other_types=outros_tipos.value, query=busca.value)
    estado["dashboard"] = resultado
    with painel:
        clear_output(wait=True)
        display(HTML("<h2>Leituras automáticas da amostra</h2><ul>" +
                     "".join("<li>" + __import__("html").escape(s) + "</li>" for s in resultado["observations"]) + "</ul>"))
        display(HTML("<h3>Qualidade e disponibilidade por fonte</h3>"))
        display(resultado["health"])
        display(HTML("<h3>Comparação de temas</h3>"))
        display(resultado["topics"])
        for grafico in resultado["figures"]:
            grafico.show()
        display(HTML("<h3>Registros recentes da seleção</h3>"))
        display(resultado["recent"][["published_date", "title", "country", "source", "record_type", "url"]].head(30))
        display(HTML("<h3>Evidências para leitura qualitativa</h3>"))
        display(resultado["evidence"].head(40))

def exportar(_=None):
    resultado = estado.get("dashboard") or agent.build_dashboard(dados, diagnosticos)
    arquivos = agent.export_results(dados, diagnosticos, resultado, SAIDA, qualidade)
    print("Resultados atualizados:")
    for nome, caminho in arquivos.items():
        print("-", caminho)
    if SALVAR_RESULTADOS_NO_DRIVE and servico_drive:
        try:
            from googleapiclient.http import MediaFileUpload
            nome_pasta = "Resultados - agente FAPESP"
            q = f"'{PASTA_DRIVE}' in parents and name = '{nome_pasta}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
            achados = servico_drive.files().list(q=q, fields="files(id,name)", pageSize=10).execute().get("files", [])
            pasta = achados[0]["id"] if achados else servico_drive.files().create(
                body={"name": nome_pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [PASTA_DRIVE]},
                fields="id").execute()["id"]
            for nome, caminho in arquivos.items():
                q = f"'{pasta}' in parents and name = '{nome}' and trashed = false"
                encontrados = servico_drive.files().list(q=q, fields="files(id,name)", pageSize=10).execute().get("files", [])
                tipo = "text/html" if nome.endswith(".html") else "application/json" if nome.endswith(".json") else "text/csv"
                media = MediaFileUpload(str(caminho), mimetype=tipo, resumable=False)
                if encontrados:
                    salvo = servico_drive.files().update(fileId=encontrados[0]["id"], media_body=media,
                        fields="id,name,webViewLink").execute()
                else:
                    salvo = servico_drive.files().create(body={"name": nome, "parents": [pasta]},
                        media_body=media, fields="id,name,webViewLink").execute()
                print("Drive:", salvo.get("webViewLink", f"arquivo {salvo['id']}"))
        except Exception as exc:
            print(f"Não foi possível salvar todos os arquivos no Drive: {exc}")
    elif SALVAR_RESULTADOS_NO_DRIVE:
        print("Sem conexão com o Drive; os arquivos locais continuam disponíveis para baixar.")

botao.on_click(mostrar)
botao_exportar.on_click(exportar)
display(widgets.VBox([widgets.HBox([periodo, busca]), widgets.HBox([paises, fontes]),
                      outros_tipos, widgets.HBox([botao, botao_exportar]), painel]))
mostrar()
exportar()

## 9. Como interpretar e ampliar

- **Comparação justa:** os gráficos temáticos dividem o número de títulos que mencionam cada tema pelo total de títulos da seleção. Fontes com poucos registros devem ser lidas com cuidado.
- **Cobertura:** a coleta pode falhar por 403, 429, 502, mudanças no HTML ou páginas vazias. Confira sempre a tabela de saúde e `diagnostico.json`.
- **Datas:** gráficos temporais usam apenas a data publicada reconhecível. Uma página institucional sem data continua no CSV, mas não entra automaticamente nas séries.
- **Próximo passo da pesquisa:** abra os links da tabela de evidências e codifique o conteúdo dos documentos antes de tirar conclusões sobre a difusão de normas entre UE, Brasil e Mercosul.

Os dados completos e o painel HTML são atualizados na subpasta **Resultados - agente FAPESP**. O CSV semanal do GitHub é alimentado pela rotina automática de quatro fontes; a rodada completa das demais fontes depende da execução deste Colab.